In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

RESULTS_DIR = './results'
COMP_DIR = os.path.join(RESULTS_DIR, 'comparison')
os.makedirs(COMP_DIR, exist_ok=True)
MODELS = ['tinyllama', 'qwen25', 'xlmr', 'xlmr_lora']
print("Starting final V11-parity analysis and exporting comparison files...")

In [ ]:
# Load all data
all_transferability = []
all_indist = []
all_eff = []
all_cross = []
all_zero = []

for model in MODELS:
    t_path = os.path.join(RESULTS_DIR, model, f"{model}_transferability.csv")
    i_path = os.path.join(RESULTS_DIR, model, f"{model}_indist_metrics.csv")
    e_path = os.path.join(RESULTS_DIR, model, f"{model}_efficiency_summary.csv")
    c_path = os.path.join(RESULTS_DIR, model, f"{model}_cross_variety_results.csv")
    z_path = os.path.join(RESULTS_DIR, model, f"{model}_zero_shot_results.csv")
    
    if os.path.exists(t_path):
        df = pd.read_csv(t_path); df['model'] = model
        all_transferability.append(df)
    if os.path.exists(i_path):
        df = pd.read_csv(i_path); df['model'] = model
        all_indist.append(df)
    if os.path.exists(e_path):
        df = pd.read_csv(e_path); df['model'] = model
        all_eff.append(df)
    if os.path.exists(c_path):
        df = pd.read_csv(c_path); df['model'] = model
        all_cross.append(df)
    if os.path.exists(z_path):
        df = pd.read_csv(z_path); df['model'] = model
        all_zero.append(df)

df_transfer = pd.concat(all_transferability, ignore_index=True) if all_transferability else pd.DataFrame()
df_indist = pd.concat(all_indist, ignore_index=True) if all_indist else pd.DataFrame()
df_eff = pd.concat(all_eff, ignore_index=True) if all_eff else pd.DataFrame()
df_cross = pd.concat(all_cross, ignore_index=True) if all_cross else pd.DataFrame()
df_zero = pd.concat(all_zero, ignore_index=True) if all_zero else pd.DataFrame()
print("Data loaded.")

In [ ]:
## 1. Architecture Comparison (in-distribution Macro F1 & Sarcasm F1)
import math
print('=' * 95)
print('ARCHITECTURE COMPARISON - IN-DISTRIBUTION MACRO-F1 & SARCASM-F1')
print('=' * 95)

MODEL_DEFS = [
    ('tinyllama', 'tinyllama_macro_f1', 'tinyllama_f1_sarcasm', 'TinyLlama (LoRA)',      '#5B9BD5'),
    ('qwen25',    'qwen25_macro_f1',    'qwen25_f1_sarcasm',    'Qwen2.5-1.5B (LoRA)',   '#ED7D31'),
    ('xlmr',      'xlmr_macro_f1',      'xlmr_f1_sarcasm',      'XLM-R (Full FT)',       '#70AD47'),
    ('xlmr_lora', 'xlmr_lora_macro_f1', 'xlmr_lora_f1_sarcasm', 'XLM-R (LoRA)',          '#9B59B6'),
]

def _get_val(df_row, *keys):
    for k in keys:
        if k in df_row.index and not (isinstance(df_row[k], float) and math.isnan(df_row[k])):
            return float(df_row[k])
    return float('nan')

if not df_indist.empty and not df_cross.empty:
    arch_rows = []

    # Per-variety in-dist rows
    test_varieties = df_indist['variety'].unique().tolist()
    for test_v in test_varieties:
        row = {'test_variety': test_v}
        scores = {}
        for model_key, col_macro, col_sarc, label, _ in MODEL_DEFS:
            sub = df_indist[(df_indist['model'] == model_key) & (df_indist['variety'] == test_v)]
            if sub.empty:
                row[col_macro] = float('nan')
                row[col_sarc]  = float('nan')
            else:
                r = sub.iloc[0]
                macro = float(r['macro_f1'])
                sarc  = _get_val(r, 'f1_sarcasm', 'f1_sarcastic')
                row[col_macro] = macro
                row[col_sarc]  = sarc
                scores[label]  = macro
        if not scores:
            continue
        row['best_model'] = max(scores, key=scores.get)
        arch_rows.append(row)

    # Combined adapter -> combined test row
    combined_row = {'test_variety': 'combined'}
    combined_scores = {}
    for model_key, col_macro, col_sarc, label, _ in MODEL_DEFS:
        sub = df_cross[
            (df_cross['model'] == model_key) &
            (df_cross['variety_train'] == 'combined') &
            (df_cross['variety_test'] == 'combined')
        ]
        if sub.empty:
            combined_row[col_macro] = float('nan')
            combined_row[col_sarc]  = float('nan')
        else:
            r = sub.iloc[0]
            macro = float(r['macro_f1'])
            sarc  = _get_val(r, 'f1_sarcasm', 'f1_sarcastic')
            combined_row[col_macro] = macro
            combined_row[col_sarc]  = sarc
            combined_scores[label]  = macro
    if combined_scores:
        combined_row['best_model'] = max(combined_scores, key=combined_scores.get)
        arch_rows.append(combined_row)

    if arch_rows:
        df_arch = pd.DataFrame(arch_rows)
        df_arch.to_csv(os.path.join(COMP_DIR, 'architecture_comparison.csv'), index=False)
        print('Saved architecture_comparison.csv')
        print(df_arch.to_string(index=False))

        present = [
            (col_macro, col_sarc, label, color)
            for _, col_macro, col_sarc, label, color in MODEL_DEFS
            if any(not math.isnan(r[col_macro]) for r in arch_rows)
        ]
        n_models = len(present)
        w = 0.8 / n_models
        offsets = [(i - (n_models - 1) / 2) * w for i in range(n_models)]

        fig, axes = plt.subplots(1, 2, figsize=(15, 5))
        fig.suptitle('Architecture Comparison: In-distribution test sets', fontsize=11, fontweight='bold')
        x       = np.arange(len(arch_rows))
        xlabels = [r['test_variety'] for r in arch_rows]

        for (col_macro, col_sarc, label, color), offset in zip(present, offsets):
            for ax, col in [(axes[0], col_macro), (axes[1], col_sarc)]:
                vals = [r[col] for r in arch_rows]
                bars = ax.bar(x + offset, vals, w, label=label, color=color,
                              edgecolor='black', linewidth=0.5)
                for bar in bars:
                    h = bar.get_height()
                    if not math.isnan(h):
                        ax.text(bar.get_x() + bar.get_width() / 2, h + 0.010,
                                f'{h:.3f}', ha='center', va='bottom', fontsize=7)

        for ax, title in [(axes[0], 'Macro-F1 Score'), (axes[1], 'Sarcasm-F1 Score')]:
            ax.set_xticks(x); ax.set_xticklabels(xlabels)
            ax.set_ylabel(title); ax.set_title(title)
            ax.set_ylim(0, 1.1); ax.legend(fontsize=8); ax.grid(axis='y', alpha=0.3)

        plt.tight_layout()
        plt.savefig(os.path.join(COMP_DIR, 'architecture_comparison.png'), dpi=150, bbox_inches='tight')
        plt.show()
        print('Saved architecture_comparison.png')


In [ ]:
## 2. Transfer to en-IN Comparison
if not df_cross.empty:
    transfer_rows = []
    varieties = df_cross['variety_train'].unique()
    for train_v in varieties:
        tl = df_cross[(df_cross['model']=='tinyllama') & (df_cross['variety_train']==train_v) & (df_cross['variety_test']=='en-IN')]
        qw = df_cross[(df_cross['model']=='qwen25') & (df_cross['variety_train']==train_v) & (df_cross['variety_test']=='en-IN')]
        xl = df_cross[(df_cross['model']=='xlmr') & (df_cross['variety_train']==train_v) & (df_cross['variety_test']=='en-IN')]
        
        if tl.empty or qw.empty or xl.empty: continue
        
        tl_f1 = tl.iloc[0]['macro_f1']
        qw_f1 = qw.iloc[0]['macro_f1']
        xl_f1 = xl.iloc[0]['macro_f1']
        
        scores = {'TinyLlama': tl_f1, 'Qwen2.5': qw_f1, 'XLM-R': xl_f1}
        transfer_rows.append({
            'train_variety': train_v,
            'test_variety':  'en-IN',
            'tinyllama_f1':  tl_f1,
            'qwen25_f1':     qw_f1,
            'xlmr_f1':       xl_f1,
            'best_model':    max(scores, key=scores.get)
        })
    pd.DataFrame(transfer_rows).to_csv(os.path.join(COMP_DIR, 'transfer_to_en_in.csv'), index=False)
    print("Saved transfer_to_en_in.csv")

In [ ]:
## 3. Inference Timing Summary & Model Efficiency
if not df_eff.empty:
    timing_rows = []

    # Mapping for known models
    model_map = {
        'tinyllama': ("TinyLlama + LoRA", "LoRA", "1.1B"),
        'qwen25': ("Qwen2.5 + LoRA", "LoRA", "1.5B"),
        'xlmr': ("XLM-R", "Full fine-tune", "270M"),
        'xlmr_lora': ("XLM-R + LoRA", "LoRA", "270M")
    }

    for _, r in df_eff.iterrows():
        key = r['model'].lower()
        if key in model_map:
            m_name, adapt, approx_params = model_map[key]
        else:
            m_name, adapt = r['model'], "Unknown"
            approx_params = f"{r.get('base_model_size_gb', 0)}GB - {r.get('base_model_params', 'Unknown')}"

        # Compute avg time if not present
        avg_ms = r.get("ms_per_sample", (r["total_inference_s"] / r["n_test_samples"]) * 1000)

        # Add row
        timing_rows.append({
            "model": m_name,
            "train_variety": r["variety"],
            "test_variety": r["variety"],
            "n_test_samples": r["n_test_samples"],
            "total_inference_time_s": r["total_inference_s"],
            "avg_time_per_sample_ms": avg_ms,
            "samples_per_second": r["n_test_samples"] / r["total_inference_s"],
            "approx_params": approx_params,
            "adaptation": adapt
        })

    # Add combined category per model
    combined_rows = []
    df_temp = pd.DataFrame(timing_rows)
    for model_name, group in df_temp.groupby("model"):
        combined_row = {
            "model": model_name,
            "train_variety": "combined",
            "test_variety": "combined",
            "n_test_samples": group["n_test_samples"].sum(),
            "total_inference_time_s": group["total_inference_time_s"].sum(),
            "avg_time_per_sample_ms": (group["total_inference_time_s"].sum() / group["n_test_samples"].sum()) * 1000,
            "samples_per_second": group["n_test_samples"].sum() / group["total_inference_time_s"].sum(),
            "approx_params": group["approx_params"].iloc[0],
            "adaptation": group["adaptation"].iloc[0]
        }
        combined_rows.append(combined_row)

    timing_rows.extend(combined_rows)

    # Raw timing DataFrame
    timing_df = pd.DataFrame(timing_rows)
    display(timing_df)
    timing_df.to_csv(os.path.join(COMP_DIR, 'inference_timing_summary.csv'), index=False)
    print("Saved inference_timing_summary.csv")

    # Aggregated efficiency report
    report_df = timing_df.groupby(
        ["model", "approx_params", "adaptation"], as_index=False
    ).agg(
        mean_avg_time_ms=("avg_time_per_sample_ms", "mean"),
        std_avg_time_ms=("avg_time_per_sample_ms", "std"),
        mean_samples_per_second=("samples_per_second", "mean")
    ).sort_values("mean_avg_time_ms")
    report_df.to_csv(os.path.join(COMP_DIR, 'model_efficiency_report.csv'), index=False)
    print("Saved model_efficiency_report.csv")
    display(report_df)

    # Human-readable efficiency table
    print('=' * 75)
    print('MODEL EFFICIENCY SUMMARY')
    print('=' * 75)
    print(f"{'Model':<22} {'Params':>10}  {'Adapt':>12}  {'Adapter MB':>11}  Note")
    print('-' * 75)
    rows = [
        ('XLM-RoBERTa',     '270M',   'Full FT',  '~220 MB',  'Encoder, fastest to train'),
        ('Qwen2.5-1.5B',    '1.5B',   'LoRA',     '~4-8 MB',  'Multilingual decoder LLM'),
        ('TinyLlama-1.1B',  '1.1B',   'LoRA',     '~4-8 MB',  'English-dominant decoder LLM'),
    ]
    for model, params, adapt, mb, note in rows:
        print(f'  {model:<22} {params:>10}  {adapt:>12}  {mb:>11}  {note}')
    print('=' * 75)

    # Per-variety inference comparison including combined
    print('\nInference time (ms/sample) - in-distribution test sets:')
    print('-' * 55)
    varieties = CONFIG.get('varieties', []) + ['combined']
    for test_v in varieties:
        tl = next((r for r in timing_rows if r["model"].startswith("TinyLlama") and r["train_variety"]==test_v), None)
        xl = next((r for r in timing_rows if r["model"].startswith("XLM-R") and r["train_variety"]==test_v), None)
        qw = next((r for r in timing_rows if r["model"].startswith("Qwen2.5") and r["train_variety"]==test_v), None)
        parts = []
        for label, r in [('TinyLlama', tl), ('XLM-R', xl), ('Qwen2.5', qw)]:
            if r:
                parts.append(f'{label}: {r["avg_time_per_sample_ms"]:.1f} ms')
        if parts:
            print(f'  {test_v}: ' + '  |  '.join(parts))
    print('\nArchitecture comparison complete')

In [ ]:
## 4. Perplexity / Pseudo-Perplexity Comparison vs LoRA Gain
import json, math, glob as _glob
import torch
import numpy as np
from tqdm.notebook import tqdm

_COMPUTE_DTYPE = torch.float16 if torch.cuda.is_available() else torch.float32

def _resolve_local_path(model_name):
    from huggingface_hub import snapshot_download
    return snapshot_download(model_name, local_files_only=True)

def _load_tokenizer_local(model_name):
    from transformers import AutoTokenizer
    local_path = _resolve_local_path(model_name)
    tok = AutoTokenizer.from_pretrained(local_path)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    return tok

def _load_causal_model(model_name):
    from transformers import AutoModelForCausalLM
    local_path = _resolve_local_path(model_name)
    return AutoModelForCausalLM.from_pretrained(
        local_path, torch_dtype=_COMPUTE_DTYPE,
        device_map='auto' if torch.cuda.is_available() else None,
    )

def _load_masked_model(model_name):
    from transformers import AutoModelForMaskedLM
    local_path = _resolve_local_path(model_name)
    return AutoModelForMaskedLM.from_pretrained(
        local_path, torch_dtype=_COMPUTE_DTYPE,
        device_map='auto' if torch.cuda.is_available() else None,
    )

def _eval_causal_perplexity(mdl, tok, texts, max_samples=200, max_length=128, desc=''):
    """Standard perplexity for auto-regressive (causal) LMs."""
    mdl.eval()
    device = next(mdl.parameters()).device
    total_loss, total_tokens = 0.0, 0
    with torch.no_grad():
        for text in tqdm(texts[:max_samples], desc=desc, leave=False):
            enc = tok(text, return_tensors='pt', max_length=max_length, truncation=True).to(device)
            if enc['input_ids'].shape[1] < 2:
                continue
            out = mdl(**enc, labels=enc['input_ids'])
            n = enc['input_ids'].shape[1]
            total_loss   += out.loss.item() * n
            total_tokens += n
    avg_loss = total_loss / total_tokens if total_tokens > 0 else float('nan')
    return round(float(np.exp(avg_loss)), 2)
    
def _find_base_model(adapter_dir, label, fallback):
    import glob as _glob
    patterns = [
        os.path.join(adapter_dir, '*', 'adapter_config.json'),
        os.path.join(adapter_dir, 'adapter_config.json'),
    ]
    for pat in patterns:
        matches = sorted(_glob.glob(pat))
        if matches:
            with open(matches[0]) as f:
                cfg = json.load(f)
            base = cfg.get('base_model_name_or_path', '')
            if base:
                print(f'  [{label}] base resolved: {base}')
                return base
    print(f'  [{label}] WARNING: using fallback: {fallback}')
    return fallback

_MODEL_CANDIDATES = [
    ('./tinyllama_adapters', 'TinyLlama', 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'),
    ('./qwen_adapters', 'Qwen2.5', 'Qwen/Qwen2.5-1.5B'),
]

CAUSAL_MODELS = []
for adapter_dir, label, fallback in _MODEL_CANDIDATES:
    base = _find_base_model(adapter_dir, label, fallback)
    CAUSAL_MODELS.append((base, label))

all_labels = [l for _, l in CAUSAL_MODELS]

from datasets import load_from_disk

ppl_path = os.path.join(COMP_DIR, 'perplexity_comparison.csv')

if not df_cross.empty and not df_zero.empty:
    varieties = [v for v in df_zero['variety_test'].unique().tolist()]

    print('Loading test texts from cached datasets...')
    variety_texts = {}
    decode_tok = _load_tokenizer_local('TinyLlama/TinyLlama-1.1B-Chat-v1.0')
    for v in varieties:
        try:
            ds = load_from_disk(f'cached_tokenized_datasets_tinyllama/{v}_test')
            variety_texts[v] = [decode_tok.decode(ids, skip_special_tokens=True)
                                for ids in ds['input_ids']]
            print(f'  [{v}] {len(variety_texts[v])} samples loaded')
        except Exception as e:
            print(f'  [{v}] skipped - {e}')

    if not variety_texts:
        print('No variety texts could be loaded.')
    else:
        if os.path.exists(ppl_path):
            print("Loading existing perplexity results...")
            df_ppl = pd.read_csv(ppl_path)
        else:
            print("Computing perplexity from scratch...")
            
            ppl_scores = {}

            for model_name, label in CAUSAL_MODELS:
                ppl_scores[label] = {}
                print(f'\nComputing perplexity - {label}...')
                try:
                    tok = _load_tokenizer_local(model_name)
                    mdl = _load_causal_model(model_name)

                    for v, texts in variety_texts.items():
                        ppl = _eval_causal_perplexity(mdl, tok, texts, desc=f'{label} [{v}]')
                        ppl_scores[label][v] = ppl
                        print(f'  [{v}] PPL = {ppl:.2f}')

                except Exception as e:
                    print(f'  Failed to load {model_name}: {e}')
                    for v in variety_texts:
                        ppl_scores[label][v] = float('nan')

                finally:
                    try: del mdl
                    except: pass
                    if torch.cuda.is_available():
                        torch.cuda.empty_cache()

        if 'ppl_scores' in locals():
            # build dataframe
            all_labels = [l for _, l in CAUSAL_MODELS]
            perplexity_rows = [
                {'variety': v, **{label: ppl_scores[label].get(v, float('nan')) for label in all_labels}}
                for v in variety_texts
            ]

            df_ppl = pd.DataFrame(perplexity_rows)

            # SAVE
            df_ppl.to_csv(ppl_path, index=False)
            print(f"Saved perplexity to {ppl_path}")

        colors = {'TinyLlama': '#5B9BD5', 'Qwen2.5': '#ED7D31',}
        model_key_map = {'TinyLlama': 'tinyllama', 'Qwen2.5': 'qwen25'}
        
        labels_present = [l for l in all_labels if l in df_ppl.columns]
        v_list = df_ppl['variety'].tolist()
        x = np.arange(len(v_list))
        n = len(labels_present)
        w = 0.7 / n

        fig, axes = plt.subplots(1, 2, figsize=(13, 4))
        fig.suptitle('Perplexity of Base Models vs LoRA Gain per Variety', fontsize=11, fontweight='bold')

        for i, label in enumerate(labels_present):
            offset = (i - (n - 1) / 2) * w
            bars = axes[0].bar(x + offset, df_ppl[label], w, label=label, color=colors.get(label, '#999'), edgecolor='black')
            for bar in bars:
                h = bar.get_height()
                if not math.isnan(h):
                    axes[0].text(bar.get_x() + bar.get_width() / 2, h + 0.5, f'{h:.1f}', ha='center', va='bottom', fontsize=8)

        axes[0].set_xticks(x); axes[0].set_xticklabels(v_list)
        axes[0].set_ylabel('Perplexity (lower = more familiar)')
        axes[0].set_title('Base Model Perplexity per Variety\n')
        axes[0].legend(fontsize=8); axes[0].grid(axis='y', alpha=0.3)

        for i, label in enumerate(labels_present):
            key = model_key_map[label]
            gains = []
            for v in v_list:
                zs_col = 'variety_test' if 'variety_test' in df_zero.columns else 'variety'
                zs = df_zero[(df_zero['model'] == key) & (df_zero[zs_col] == v)]
                indist = df_indist[(df_indist['model'] == key) & (df_indist['variety'] == v)]
                if zs.empty or indist.empty:
                    print(f"[WARNING] Missing data for {key} - {v}")
                    gains.append(np.nan)
                else:
                    gains.append(indist['macro_f1'].values[0] - zs['macro_f1'].values[0])

            # compute combined gain from cross-variety
            if 'combined' in v_list:
                mask = df_cross['variety_test'] != 'combined'
                df_cross_subset = df_cross.loc[mask]

                # get zero-shot F1 per test variety
                zs_f1_aligned = []
                for _, row in df_cross_subset.iterrows():
                    v_test = row['variety_test']
                    zs_row = df_zero[(df_zero['variety_test'] == v_test) & (df_zero['model'] == key)]
                    if zs_row.empty:
                        zs_f1_aligned.append(np.nan)
                    else:
                        zs_f1_aligned.append(zs_row['macro_f1'].values[0])

                f1_diffs = df_cross_subset['macro_f1'].values - np.array(zs_f1_aligned)
                weights = df_cross_subset['n_test_samples'].values
                combined_gain = np.average(f1_diffs, weights=weights)

                # update gains for combined
                combined_index = v_list.index('combined')
                gains[combined_index] = combined_gain

            # DEBUG OUTPUT
            print(f"{label} gains per variety:", list(zip(v_list, gains)))

            offset = (i - (n - 1) / 2) * w
            bars = axes[1].bar(x + offset, gains, w, label=f'{label} gain', color=colors.get(label, '#999'), edgecolor='black')
            for bar in bars:
                axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005, f'{bar.get_height():+.3f}', ha='center', va='bottom', fontsize=8)

        axes[1].axhline(0, color='black', linewidth=0.8)
        axes[1].set_xticks(x); axes[1].set_xticklabels(v_list)
        axes[1].set_ylabel('Macro-F1 gain over zero-shot / base')
        axes[1].set_title('Adapter Gain Over Baseline')
        axes[1].legend(fontsize=8); axes[1].grid(axis='y', alpha=0.3)

        plt.tight_layout()
        plt.savefig(os.path.join(COMP_DIR, 'perplexity_vs_lora_gain.png'), dpi=150, bbox_inches='tight')
        plt.show()
        print('Saved perplexity_vs_lora_gain.png')

print('\n--- Perplexity / PLL Analysis Complete ---')


In [ ]:
## 5. Accuracy Across All Models and Varieties
import math

print('=' * 80)
print('ACCURACY COMPARISON ACROSS ALL MODELS AND VARIETIES')
print('=' * 80)

MODEL_COLORS = {
    'tinyllama':  '#5B9BD5',
    'qwen25':     '#ED7D31',
    'xlmr':       '#70AD47',
    'xlmr_lora':  '#9B59B6',
}
MODEL_LABELS = {
    'tinyllama':  'TinyLlama (LoRA)',
    'qwen25':     'Qwen2.5 (LoRA)',
    'xlmr':       'XLM-R (Full FT)',
    'xlmr_lora':  'XLM-R (LoRA)',
}

if not df_indist.empty and not df_cross.empty:
    acc_rows = []
    test_varieties = sorted(df_indist['variety'].unique().tolist())
    for v in test_varieties:
        row = {'variety': v}
        for model_key in MODELS:
            sub = df_indist[(df_indist['model'] == model_key) & (df_indist['variety'] == v)]
            row[model_key] = float(sub.iloc[0]['accuracy']) if not sub.empty else float('nan')
        acc_rows.append(row)

    combined_row = {'variety': 'combined'}
    for model_key in MODELS:
        sub = df_cross[
            (df_cross['model'] == model_key) &
            (df_cross['variety_train'] == 'combined') &
            (df_cross['variety_test'] == 'combined')
        ]
        combined_row[model_key] = float(sub.iloc[0]['accuracy']) if not sub.empty else float('nan')
    acc_rows.append(combined_row)

    df_acc = pd.DataFrame(acc_rows)
    df_acc.to_csv(os.path.join(COMP_DIR, 'accuracy_all_models.csv'), index=False)
    print('Saved accuracy_all_models.csv')

    print(f"\n{'Variety':<12}", end='')
    for m in MODELS:
        print(f"  {MODEL_LABELS[m]:>20}", end='')
    print()
    print('-' * (12 + 22 * len(MODELS)))
    for _, r in df_acc.iterrows():
        print(f"  {r['variety']:<10}", end='')
        for m in MODELS:
            val = r[m]
            print(f"  {val:>20.4f}" if not math.isnan(val) else f"  {'N/A':>20}", end='')
        print()

    x = np.arange(len(acc_rows))
    n = len(MODELS)
    w = 0.8 / n
    xlabels = [r['variety'] for r in acc_rows]

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.set_title(
        'Accuracy Across All Models and Varieties\n'
        '(in-dist for per-variety; combined adapter for "combined")',
        fontsize=11, fontweight='bold')

    for i, model_key in enumerate(MODELS):
        offset = (i - (n - 1) / 2) * w
        vals = [r[model_key] for r in acc_rows]
        bars = ax.bar(x + offset, vals, w,
                      label=MODEL_LABELS[model_key],
                      color=MODEL_COLORS[model_key],
                      edgecolor='black', linewidth=0.5)
        for bar in bars:
            h = bar.get_height()
            if not math.isnan(h):
                ax.text(bar.get_x() + bar.get_width() / 2, h + 0.008,
                        f'{h:.3f}', ha='center', va='bottom', fontsize=7)

    ax.set_xticks(x); ax.set_xticklabels(xlabels)
    ax.set_ylabel('Accuracy')
    ax.set_ylim(0, 1.12)
    ax.legend(fontsize=9); ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(COMP_DIR, 'accuracy_all_models.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved accuracy_all_models.png')
